# 低功耗无线温湿度节点

## 项目简介

本项目实现一个基于 ESP32、DHT22 和电池供电的低功耗无线温湿度节点。节点定时唤醒采集温湿度和电池电压，通过 ESP-NOW 广播方式发送到网关端，随后立即进入深度睡眠，以延长电池工作时间，适合作为分布式环境监测终端原型。


## 主要器件

| 器件 | 数量 | 说明 |
| --- | --- | --- |
| ESP32 | 1 | 主控与无线通信 |
| DHT22 | 1 | 温湿度采集 |
| 电池与分压电路 | 1 套 | 供电与电压检测 |
| 网关端 ESP32 | 选配 | 接收广播数据 |


## 核心功能

- 周期唤醒并采集温湿度。
- 读取电池电压用于评估节点供电状态。
- 使用 ESP-NOW 广播发送采样数据。
- 发送后进入深度睡眠，降低平均功耗。
- 通过 RTC 存储唤醒计数，便于确认节点运行节奏。


## 引脚分配

| 模块 | 引脚 |
| --- | --- |
| DHT22 | GPIO4 |
| 电池电压检测 | GPIO34 |
| 无线发送 | ESP32 内置 WiFi / ESP-NOW |


## 接线说明

- DHT22 数据脚接 GPIO4。
- 电池电压通过分压网络接到 GPIO34，分压比需与代码中的换算一致。
- 若节点由锂电池供电，应确保深度睡眠期间外围模块也尽量降低功耗。
- 网关端需使用相同结构的 `Payload` 解析广播数据。


## 运行方式

1. 上传 `src/low_power_wireless_temperature_humidity_node/low_power_wireless_temperature_humidity_node.ino` 到 ESP32。
2. 连接 DHT22 和电池电压采样分压电路。
3. 准备另一个 ESP32 或上位机网关接收 ESP-NOW 数据。
4. 上电后观察节点发送一次数据并进入深度睡眠。
5. 根据部署周期调整 `SLEEP_SECONDS`。


## 控制逻辑说明

- 节点每次唤醒后只完成一次采样和一次发送，不在主循环内长期驻留。
- 采样结果被打包到 `Payload` 结构中，包含节点编号、温湿度、电压和唤醒次数。
- 发送采用广播方式，方便先把节点和网关链路打通。
- 发送结束后进入深度睡眠，显著降低平均功耗。
- 若传感器读数失败，则使用明显异常值，方便网关判断节点健康状态。


## 验证标准

| 测试项 | 通过标准 |
| --- | --- |
| 温湿度采集 | 唤醒后能得到有效温湿度 |
| 无线发送 | 网关端可收到完整负载数据 |
| 电池检测 | 负载中包含电池电压字段 |
| 深度睡眠 | 发送后节点进入睡眠并周期性唤醒 |


## 可扩展方向

- 更换为 LoRa、nRF24 等远距离无线方案。
- 增加太阳能供电和充电管理。
- 增加多节点网关聚合与本地缓存。
- 增加异常重发和链路质量统计。
